In [17]:
import jax
import jax.numpy as jnp
import os, json, gzip, glob
import pandas as pd
from functools import wraps

# The secret sauce: Annotation + Sync
def traceable(name):
    def decorator(fn):
        @wraps(fn)
        def wrapper(*args, **kwargs):
            # 1. Create a label the profiler can see
            with jax.profiler.TraceAnnotation(name):
                # 2. Run the actual JAX function
                result = fn(*args, **kwargs)
                # 3. Force GPU to finish so the 'box' in the trace is the right size
                return result.block_until_ready()
        return wrapper
    return decorator

In [19]:
@traceable(name="PART_1_MATRIX_MULT")
@jax.jit
def foo_1(x):
    return jnp.dot(x, x)

@traceable(name="PART_2_EXPONENTIAL")
@jax.jit
def foo_2(x):
    # We do a few more operations to make it stand out on the GPU
    return jnp.exp(-jnp.abs(x)) + jnp.sin(x)

# Main entry point (Un-JITted)
def foo(x):
    res1 = foo_1(x)
    res2 = foo_2(res1)
    return jnp.mean(res2)

In [20]:
# Create storage for trace
trace_dir = "./final_scratch_trace"
os.makedirs(trace_dir, exist_ok=True)

# Generate data (Large enough for GPU to actually work)
x_input = jnp.ones((4096, 4096))

print("Step 1: Warming up...")
_ = foo(x_input)

print("Step 2: Capturing Trace...")
jax.profiler.start_trace(trace_dir)
# We run it 5 times to get better data
for _ in range(5):
    _ = foo(x_input)
jax.profiler.stop_trace()
print("Done.")

Step 1: Warming up...
Step 2: Capturing Trace...
Done.


In [21]:
def extract_results(path, csv_name="jax_timings.csv"):
    # Find the trace file
    files = glob.glob(os.path.join(path, "**/*.json.gz"), recursive=True)
    if not files:
        print("No trace found!")
        return

    with gzip.open(files[0], 'rb') as f:
        events = json.load(f).get("traceEvents", [])

    raw_data = []
    targets = ["PART_1_MATRIX_MULT", "PART_2_EXPONENTIAL"]

    for e in events:
        name = str(e.get("name", ""))
        for t in targets:
            if t in name:
                dur = e.get("dur")
                if dur:
                    raw_data.append({"Function": t, "Duration_ms": dur / 1000})

    # Create DataFrame and aggregate stats
    df = pd.DataFrame(raw_data)
    if not df.empty:
        # Get Average and Std Dev
        stats = df.groupby("Function")["Duration_ms"].agg(['mean', 'std', 'max']).reset_index()
        stats.to_csv(csv_name, index=False)
        print("\n--- RESULTS ---")
        print(stats)
    else:
        print("No data found in trace.")

extract_results(trace_dir)


--- RESULTS ---
             Function      mean       std       max
0  PART_1_MATRIX_MULT  0.751136  0.288616  1.209310
1  PART_2_EXPONENTIAL  0.309894  0.133400  0.504234
